In [ ]:
import os
import sys

module_path = os.path.abspath(os.path.join("..", "src", "py_gis_library_database"))

if module_path not in sys.path:
    sys.path.append(module_path)

In [ ]:
import matplotlib.pyplot as plt
import mercantile
import numpy as np
import plotly.graph_objects as go
from climb_detector import detect_significant_segments
from data_fun import (
    get_bounds_from_gdf,
    read_trk_file_add_additional_properties,
    use_dem_model,
)
from general_fun import ceil_date, floor_date, rounddown, roundup
from map_fun import getMap
from matplotlib.gridspec import GridSpec
from plot_fun import draw_detailed_colored_profile
from plotly.subplots import make_subplots


In [ ]:
plt.rcParams.update({"font.family": "sans-serif"})
plt.rcParams["font.size"] = 8
plt.rc("font", size=8)  # controls default text sizes
plt.rc("axes", titlesize=8)  # fontsize of the axes title
plt.rc("axes", labelsize=8)  # fontsize of the x and y labels
plt.rc("xtick", labelsize=8)  # fontsize of the tick labels
plt.rc("ytick", labelsize=8)  # fontsize of the tick labels
plt.rc("legend", fontsize=8)  # legend fontsize
plt.rc("figure", titlesize=10)  # fontsize of the figure title

In [ ]:
gpx_store = "/home/wcn/SynologyDrive/19. GPS/19.10_Hikes/AOT/2026/"

nh = 3

trk_files = [
    "2026-06-11_102338--knivskjelodden.trk",  # 0
    "2026-06-14_100013--LOF_28_festvagtind.trk",  # 1
    "2026-06-16_094425--LOF_26_vetten.trk",  # 2
    "2026-06-18_070943--Mannen_HR.trk",  # 3
    "2026-06-24_095639--fottur-til-maskat-687-moh-fra-roheim-trip.trk",  # 4
]

names = ["Knivskjelodden", "Festvagtind", "Vetten", "Mannen", "Maskat"]

gpx_name = names[nh]
gpx_file = trk_files[nh]
gpx_filepath = os.path.join(".", gpx_file)

cacheDir = os.path.join(os.path.expanduser("~"), ".map_cache")
dem_folder = os.path.join(os.path.expanduser("~"), ".map_cache", "dem")

useCache = True
maxAgeDays = 180

bound_value = 0.01

zoom_level = 13

params = {
    "max_pause_length_m": 100,
    "max_pause_descent_m": 5,
    "start_threshold_slope": 3.0,
}

In [ ]:
gdf, stats = read_trk_file_add_additional_properties(gpx_filepath, max_speed_mps=4)

print(f"CRS: {gdf.crs}")
print(f"time dtype: {gdf['time'].dtype}")
print(f"Start (UTC): {gdf['time'].iloc[0]}")
print(f"End   (UTC): {gdf['time'].iloc[-1]}")

print("\nStats")
for key, value in stats.items():
    print(f"{key:<20}: {value:>15,.2f}")

In [ ]:
gdf, dem_array, transform = use_dem_model(
    gdf,
    dem_model_tool_use="dem_stitcher",
    dem_name="glo_30",
    dem_folder=dem_folder,
)

In [ ]:
climbs_df = detect_significant_segments(gdf, kind="climb", **params)
descents_df = detect_significant_segments(gdf, kind="descent", **params)

In [ ]:
minx, miny, maxx, maxy = bounds = get_bounds_from_gdf(gdf, bound_value=bound_value)

tiles = list(mercantile.tiles(minx, miny, maxx, maxy, zoom_level))
qkeys = [mercantile.quadkey(*tile) for tile in tiles]

img, img_bb = getMap(
    server_name="opentopomap",
    maptype="",
    qkeys=qkeys,
    cacheDir=cacheDir,
    useCache=useCache,
    maxAgeDays=maxAgeDays,
)
img_extend = (img_bb.west, img_bb.east, img_bb.south, img_bb.north)

In [ ]:
xvalues1 = np.array(gdf["time"])
xvalues2 = np.array(gdf["distance"])

yvalues1 = np.array(gdf["elevation"])
yvalues2 = np.array(gdf["elevation_dem"])

ymax = roundup(np.max([np.max(yvalues1), np.max(yvalues2)]), 250)

xmind = 0
xmaxd = roundup(np.max(xvalues2), 250)

xmint = floor_date(np.min(xvalues1), minutes=30)
xmaxt = ceil_date(np.max(xvalues1), minutes=30)

Zi = np.flipud(dem_array)

nrows, ncols = Zi.shape

x = np.linspace(minx, maxx, ncols)
y = np.linspace(miny, maxy, nrows)
Xi, Yi = np.meshgrid(x, y)

vmin = np.max([0, rounddown(dem_array.min(), 100)])
vmax = roundup(dem_array.max(), 100)

step = 100

extent = minx, maxx, miny, maxy

end = rounddown(gdf["distance"].max(), 1000)
kp = np.arange(1000, end + 1, 1000)

ch = [
    np.where(np.abs(gdf["distance"] - x) == np.min(np.abs(gdf["distance"] - x)))[0][0]
    for x in kp
]
kp_markers = gdf.loc[ch, ["geometry"]].reset_index(drop=True).copy()
kp_markers["labels"] = (kp / 1000).astype(int)

In [ ]:
le = f"total distance: {stats['total_distance_km']:.3f} km"
eg = f"elevation gain: {stats['elevation_gain']:.1f} m"
el = f"elevation loss: {stats['elevation_loss']:.1f} m"
mt = f"moving time: {stats['moving_time_min']:.1f} min"
tt = f"total time: {stats['total_time_min']:.1f} min"

fig = plt.figure(figsize=(11.69, 8.27))

gs = GridSpec(2, 2, figure=fig)

ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])
ax3 = fig.add_subplot(gs[1, :])

# fig 0,0
gdf.plot(ax=ax1, markersize=1, zorder=11, color="red")
kp_markers.plot(ax=ax1, markersize=5, zorder=12, color="k")
for coords, label in zip(kp_markers["geometry"], kp_markers["labels"], strict=True):
    ax1.annotate(
        str(label),
        (coords.x, coords.y),
        xytext=(0, 2),
        textcoords="offset pixels",
        fontsize=6,
        ha="right",
        va="bottom",
        color="k",
        zorder=13,
    )

ax1.imshow(img, extent=img_extend)

ax1.set(
    xlabel="longitude [deg]",
    xlim=[minx, maxx],
    xticks=np.arange(minx, maxx + bound_value / 2, bound_value),
    ylabel="latitude [deg]",
    ylim=[miny, maxy],
    yticks=np.arange(miny, maxy + bound_value / 2, bound_value),
    aspect="equal",
)

ax1.grid(True)

ax1.ticklabel_format(useOffset=False, style="plain")

ax1.set_title(
    "Topographic Chart",
    fontweight="bold",
    fontsize=8,
)

# fig 0,1
cax = ax2.imshow(dem_array, extent=extent, cmap="terrain", vmin=vmin, vmax=vmax)

ax2.contour(
    dem_array,
    extent=extent,
    levels=np.arange(vmin, vmax + step, step),
    colors="black",
    linewidths=0.5,
)

gdf.plot(ax=ax2, markersize=1, zorder=11, color="red")
kp_markers.plot(ax=ax2, markersize=5, zorder=12, color="k")
for coords, label in zip(kp_markers["geometry"], kp_markers["labels"], strict=True):
    ax2.annotate(
        str(label),
        (coords.x, coords.y),
        xytext=(0, 2),
        textcoords="offset pixels",
        fontsize=6,
        ha="right",
        va="bottom",
        color="k",
        zorder=13,
    )

ax2.set(
    xlabel="longitude [deg]",
    xlim=[minx, maxx],
    xticks=np.arange(minx, maxx + bound_value / 2, bound_value),
    ylabel="latitude [deg]",
    ylim=[miny, maxy],
    yticks=np.arange(miny, maxy + bound_value / 2, bound_value),
    aspect="equal",
)

ax2.grid(True)

ax2.ticklabel_format(useOffset=False, style="plain")

cbax = fig.add_axes(
    [
        ax2.get_position().x1 + 0.005,
        ax2.get_position().y0,
        0.01,
        ax2.get_position().height,
    ]
)
cbar = fig.colorbar(cax, cax=cbax)
cbar.set_label("elevation [m]")
cbar.ax.set_yticks(np.arange(vmin, vmax + step, step))

ax2.set_title(
    "Digital Elevation Model",
    fontweight="bold",
    fontsize=8,
)

# fig 1,0:1
draw_detailed_colored_profile(
    ax3,
    gdf,
    climbs_df,
    descents_df,
    color_by_slope=True,
    show_markers=True,
    color_mode="Detailed Slope",
)

ax3.set(
    xlabel="distance [km]",
    ylabel="elevation [m]",
    ylim=[0, roundup(np.max(gdf["elevation"]), 250)],
    xlim=[-0.05, roundup(np.max(gdf["distance"]) / 1000, 1)],
    xticks=np.arange(0, roundup(np.max(gdf["distance"]) / 1000, 1) + 0.5, 1),
)
ax3.grid(True)

ax3.set_title(
    f"Elevation Profile [{eg}, {el}]",
    fontweight="bold",
    fontsize=8,
)

fig.suptitle(
    f"Hike - {gpx_name}\n{gdf['time'].iloc[0].strftime('%Y-%m-%d %H:%M')}, {le}, {mt}, {tt}",
    fontweight="bold",
    fontsize=10,
)

fig.savefig(
    os.path.join(
        "img",
        f"{gdf['time'].iloc[0].strftime('%Y%m%d')}_overview_{gpx_name.lower()}.png",
    ),
    dpi=600,
    transparent=True,
)
plt.show()

In [ ]:
fig = go.Figure(
    make_subplots(
        rows=2,
        cols=2,
        specs=[
            [{"type": "xy"}, {"type": "map"}],
            [{"type": "xy"}, {"type": "scene"}],
        ],
        shared_xaxes=False,
        shared_yaxes=False,
        vertical_spacing=0.125,
        horizontal_spacing=0.025,
        row_heights=[0.40, 0.40],
        column_widths=[0.75, 0.25],
        subplot_titles=(
            "Elevation profile in time",
            "Chart",
            "Elevation profile in distance",
            "3D Elevation profile",
        ),
    ),
)

fig.add_trace(
    go.Scatter(
        x=xvalues1,
        y=yvalues1,
        mode="lines",
        name="elevation",
        legendgroup="elevation",
        line_color="blue",
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=xvalues1,
        y=yvalues2,
        mode="lines",
        name="elevation from dem",
        legendgroup="elevation from dem",
        line_color="purple",
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scattermap(
        lat=gdf["geometry"].y,
        lon=gdf["geometry"].x,
        mode="lines",
        line={"width": 1, "color": "red"},
        name="track",
        below="''",
        # hoverinfo="skip",
        legendgroup="track",
    ),
    row=1,
    col=2,
)
fig.add_trace(
    go.Scattermap(
        lat=kp_markers["geometry"].y,
        lon=kp_markers["geometry"].x,
        mode="markers",
        line={"width": 5, "color": "black"},
        name="chainage",
        below="''",
        # hoverinfo="skip",
        legendgroup="chainage",
    ),
    row=1,
    col=2,
)

fig.add_trace(
    go.Scatter(
        x=xvalues2,
        y=yvalues1,
        mode="lines",
        name="elevation",
        legendgroup="elevation",
        line_color="blue",
    ),
    row=2,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=xvalues2,
        y=yvalues2,
        mode="lines",
        name="elevation from dem",
        legendgroup="elevation from dem",
        line_color="purple",
    ),
    row=2,
    col=1,
)

fig.add_trace(
    go.Surface(
        x=Xi,
        y=Yi,
        z=Zi,
        colorscale="earth_r",
        cmin=vmin,
        cmax=vmax,
        contours={
            "z": {
                "show": True,
                "usecolormap": True,
                "highlightcolor": "red",
                "project": {"z": True},
            }
        },
        colorbar={"x": 1.005, "y": 0.215, "len": 0.45, "title": "elevation [m]"},
    ),
    row=2,
    col=2,
)


fig.add_trace(
    go.Scatter3d(
        x=gdf["geometry"].x,
        y=gdf["geometry"].y,
        z=yvalues2,
        name="track (elevation from dem)",
        mode="markers",
        marker={
            "size": 5,
            "color": yvalues2,
            "colorscale": "portland",
        },
        legendgroup="3D",
    ),
    row=2,
    col=2,
)


Yaxis = {
    "title": "elevation [m]",
    "range": [0, ymax],
    "tickmode": "linear",
    "ticks": "outside",
    "tick0": 0,
    "linewidth": 1,
    "mirror": True,
    "dtick": 50,
    "tickfont": {"size": 12},
}

Xaxis = {
    "ticks": "outside",
    "linewidth": 1,
    "mirror": True,
    "tickfont": {"size": 12},
}

lat_axis = {
    "title": "latitude [deg]",
    "range": [miny, maxy],
    "tickmode": "linear",
    "ticks": "outside",
    "tick0": miny,
    "linewidth": 1,
    "mirror": True,
    "dtick": bound_value,
    "tickfont": {"size": 12},
}

lon_axis = {
    "title": "longitude [deg]",
    "range": [minx, maxx],
    "tickmode": "linear",
    "ticks": "outside",
    "tick0": minx,
    "linewidth": 1,
    "mirror": True,
    "dtick": bound_value,
    "tickfont": {"size": 12},
}

fig.update_scenes(
    xaxis=lon_axis,
    yaxis=lat_axis,
    zaxis=Yaxis,
)

fig.update_layout(
    xaxis1=Xaxis
    | {
        "title": "time (UTC) [s]",
        "range": [xmint, xmaxt],
        "tick0": xmint,
        "dtick": 15 * 60 * 1000,
    },
    xaxis2=Xaxis
    | {
        "title": "distance [m]",
        "tickmode": "linear",
        "tick0": 0,
        "dtick": 500,
        "range": [xmind, xmaxd],
    },
    yaxis1=Yaxis,
    yaxis2=Yaxis,
    map={
        "style": "open-street-map",
        "center": {
            "lat": gdf["geometry"].y.median(),
            "lon": gdf["geometry"].x.median(),
        },
        "zoom": zoom_level,
        "layers": [
            {
                "below": "traces",
                "sourcetype": "raster",
                "sourceattribution": 'Map style: © <a href="https://opentopomap.org">OpenTopoMap</a> (<a href="https://creativecommons.org/licenses/by-sa/3.0/">CC-BY-SA</a>)',
                "source": [
                    "https://a.tile.opentopomap.org/{z}/{x}/{y}.png",
                    "https://b.tile.opentopomap.org/{z}/{x}/{y}.png",
                    "https://c.tile.opentopomap.org/{z}/{x}/{y}.png",
                ],
            }
        ],
    },
    margin={"l": 75, "r": 10, "b": 10, "t": 50},
    template="plotly_dark",
    font={"color": "white", "family": "Arial", "size": 14},
    hovermode="x unified",
    showlegend=True,
    height=1000,
    title=f"Hike - {gpx_name}",
)
fig.show()